In [1]:
!uv pip install -q "sagemaker>=2.0.0,<3.0.0" "sagemaker[local]" "boto3==1.42.82" "pytz==2026.1.post1" "requests==2.33.1"

In [2]:
!uv pip freeze | grep -E "sagemaker|boto3|pytz|requests"

boto3==1.42.82
pytz==2026.1.post1
requests==2.33.1
sagemaker==2.257.1
sagemaker-core==1.0.77


In [3]:
import os
import subprocess
import sys
from pprint import pprint

import boto3
import requests
from requests.auth import HTTPBasicAuth

In [18]:
url = "http://airflow-webserver:8080/api/v1/dags"

try:
    response = requests.get(url, auth=HTTPBasicAuth("admin", "admin"), timeout=10)

    if response.status_code == 200:
        data = response.json()
        if len(data['dags']) > 0:
            for dag in data['dags']:
                print(dag["dag_id"])
                
    else:
        print(f"Failed with status: {response.status_code}")
        print(f"Response Body: {response.text}")

except Exception as e:
    print(f"Connection Error: {e}")

credit_risk_data_pipeline
credit_risk_inference_pipeline


In [5]:
!docker build --no-cache -q -t credit-risk-training:latest -f /app/give_me_some_credit/sagemaker/training/Dockerfile /app/give_me_some_credit/sagemaker/training/

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/



sha256:bd7a67b4ce980d6372d65b94d1d2ee03a007e62c1bba589b2738d3ef7a7369c6


In [6]:
!aws s3 ls s3://data-lake --recursive

2026-04-03 20:28:53          0 bronze/
2026-04-03 20:33:04    7564965 bronze/credit_risk/kaggle/2026-04-03/cs-training.csv
2026-04-03 20:36:40    4983329 bronze/credit_risk/kaggle/2026-04-03/inference/cs-test.csv
2026-04-03 20:28:57          0 feast/registry/
2026-04-03 20:28:55          0 gold/
2026-04-03 20:36:07          0 gold/credit_risk/features/ingestion_date=2026-04-03/
2026-04-03 20:36:08    1662771 gold/credit_risk/features/ingestion_date=2026-04-03/part-00000-dce88d0f-8578-4968-b7b5-09e832cff668.c000.snappy.parquet
2026-04-03 20:36:08    1644117 gold/credit_risk/features/ingestion_date=2026-04-03/part-00001-dce88d0f-8578-4968-b7b5-09e832cff668.c000.snappy.parquet
2026-04-03 20:37:15          0 gold/credit_risk/inference_features/ingestion_date=2026-04-03/
2026-04-03 20:37:15    1329743 gold/credit_risk/inference_features/ingestion_date=2026-04-03/part-00000-752f5627-9adc-4fa9-82c2-40735a2c2e4a.c000.snappy.parquet
2026-04-03 20:37:15    1322762 gold/credit_risk/inference_feat

In [7]:
def get_latest_ingestion_date(bucket, prefix, s3_endpoint=None):
    s3 = boto3.client("s3", endpoint_url=s3_endpoint)

    # We use delimiter='/' to get "folders" inside CommonPrefixes
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix, Delimiter="/")

    dates = []
    for page in pages:
        for obj in page.get("CommonPrefixes", []):
            # Extract 'YYYY-MM-DD' from 'path/ingestion_date=YYYY-MM-DD/'
            folder_name = obj.get("Prefix").split("=")[-1].strip("/")
            dates.append(folder_name)

    if not dates:
        raise ValueError(f"No partitions found in s3://{bucket}/{prefix}")

    # Sorting ensures '2026-03-22' comes after '2026-03-21'
    return sorted(dates)[-1]


# --- Usage in your script ---
S3_ENDPOINT = "http://localstack:4566"
LATEST_DATE = get_latest_ingestion_date(
    bucket="data-lake", prefix="gold/credit_risk/features/", s3_endpoint=S3_ENDPOINT
)

print(f"Detected Latest Date: {LATEST_DATE}")

Detected Latest Date: 2026-04-03


In [8]:
NETWORK = os.environ["NETWORK"]
result = subprocess.run(
    [
        sys.executable,
        "/app/give_me_some_credit/sagemaker/training/sm_pipeline.py",
        "--mode",
        "local",
        "--ingestion-date",
        LATEST_DATE,
        "--s3-endpoint",
        "http://localstack:4566",
        "--n-trials",
        "3",
        "--auc-threshold",
        "0.85",
        "--network",
        NETWORK,
    ],
    capture_output=False,
    text=True,
)
print(f"\nExit code: {result.returncode}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


INFO:sagemaker_pipeline_give_me_some_credit:Args: {'mode': 'local', 'ingestion_date': '2026-04-03', 's3_endpoint': 'http://localstack:4566', 's3_bucket': 'data-lake', 'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'credit_risk_pipeline', 'n_trials': 3, 'auc_threshold': 0.85, 'training_image': 'credit-risk-training:latest', 'network': 'mlops-lab_mlops-lab-net', 'aws_region': 'us-east-1'}
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:sagemaker_session:Local SageMaker session ready (bucket=data-lake)
INFO:botocore.credentials:Found credentials in environment variables.


INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for scheduler via: environment_global.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagema

INFO:botocore.configprovider:Found endpoint for sts via: environment_global.


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.entities:Starting execution for pipeline CreditRiskTrainingPipeline. Execution ID is f2df7218-a221-497d-92d6-45cec59fd430
INFO:sagemaker.local.entities:Starting pipeline step: 'Preprocessing'
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overvie

...

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.


INFO:sagemaker.local.entities:Pipeline step 'Evaluation' SUCCEEDED.
INFO:sagemaker.local.entities:Starting pipeline step: 'CheckAUCThreshold'
INFO:sagemaker.local.entities:Pipeline step 'CheckAUCThreshold' SUCCEEDED.
INFO:sagemaker.local.entities:Pipeline execution f2df7218-a221-497d-92d6-45cec59fd430 SUCCEEDED
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.


INFO:sagemaker_pipeline_give_me_some_credit:Pipeline execution started: local
INFO:sagemaker_pipeline_give_me_some_credit:Pipeline step summary:
INFO:sagemaker_pipeline_give_me_some_credit:Could not retrieve step summary: 'str' object has no attribute 'get'
INFO:sagemaker_pipeline_give_me_some_credit:Pipeline complete.


i6gn21j91v-algo-1-62spu  | 2026-04-03 20:44:34,960 - tune_step - INFO - Uploaded tuning_summary.json => s3://data-lake/projects/credit_risk_pipeline/sagemaker/pipeline/tuning/tuning_summary.json
i6gn21j91v-algo-1-62spu  | 2026-04-03 20:44:34,961 - tune_step - INFO - Tuning complete.
i6gn21j91v-algo-1-62spu exited with code 0
 Compose Stopping Aborting on container exit...
 Container i6gn21j91v-algo-1-62spu Stopping 
 Container i6gn21j91v-algo-1-62spu Stopped 
time="2026-04-03T20:44:36Z" level=warning msg="/tmp/tmplsbfapni/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-04-03T20:44:36Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmplsbfapni\".\nSet `external: true` to use an existing network"
 Container oavfl71sxj-algo-1-2d0s8 Creating 
 Container oavfl71sxj-algo-1-2d0s8 Created 
Attaching to oavfl71sxj-algo-1-2d0s8
 Container oavfl71sxj-algo-1-2d0


Exit code: 0
